# ELIA WILD — production-gated Kaggle launch

This notebook implements one fail-closed ELIA lifecycle: restore-or-genesis → CPU viability gates → one bounded GPU cognition burst → encrypted checkpoint → independent restore proof.

Both continuity secrets are mandatory. A failed gate stops the launch instead of degrading to plaintext or unverified state.

In [ ]:
REPO_REF = 'elia/genesis-1.7.1-consolidation'
!rm -rf /kaggle/working/ELIA-WILD
!git clone --branch {REPO_REF} --single-branch https://github.com/vvseweedno/ELIA-WILD.git /kaggle/working/ELIA-WILD
%cd /kaggle/working/ELIA-WILD
!python -m pip install -q -e '.[test]'

In [ ]:
import base64, json, os, pathlib, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

STATE_DIR = pathlib.Path('/kaggle/working/elia-state')
CHECKPOINT = pathlib.Path('/kaggle/working/elia-genesis.eliacp')
DIGEST_FILE = pathlib.Path('/kaggle/working/trusted-digest.txt')
RESTORE_PROOF = pathlib.Path('/kaggle/working/elia-restore-proof')
ENVELOPE_MAGIC = b'ELIA-WILD-CHECKPOINT-ENC-v1\n'

secrets = UserSecretsClient()
auth_key = secrets.get_secret('ELIA_CHECKPOINT_KEY')
enc_key = secrets.get_secret('ELIA_CHECKPOINT_ENCRYPTION_KEY')
assert auth_key and len(auth_key) >= 16, 'ELIA_CHECKPOINT_KEY is missing/too short'
decoded = base64.b64decode(enc_key, validate=True)
assert len(decoded) == 32, 'ELIA_CHECKPOINT_ENCRYPTION_KEY must be base64 for exactly 32 bytes'

os.environ['ELIA_CHECKPOINT_KEY'] = auth_key
os.environ['ELIA_CHECKPOINT_ENCRYPTION_KEY'] = enc_key
os.environ['ELIA_CHECKPOINT_REQUIRE_ENCRYPTION'] = '1'
os.environ['ELIA_STATE_DIR'] = str(STATE_DIR)
os.environ['ELIA_AUTO_CHECKPOINT_PATH'] = str(CHECKPOINT)
def parse_cli_json(text):
    stripped = text.strip()
    for start in reversed([i for i, ch in enumerate(stripped) if ch == '{']):
        try:
            item = json.loads(stripped[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(item, dict):
            return item
    raise RuntimeError('ELIA CLI did not return a JSON object')

print('Continuity secrets loaded; encrypted mode is mandatory.')

In [ ]:
# Restore the latest attached encrypted state if one exists; otherwise create Genesis state.
input_root = pathlib.Path('/kaggle/input')
prior_checkpoints = list(input_root.rglob('elia-genesis.eliacp'))
prior_digests = list(input_root.rglob('trusted-digest.txt'))
assert len(prior_checkpoints) <= 1 and len(prior_digests) <= 1, 'Attach at most one ELIA state dataset'

if prior_checkpoints:
    assert len(prior_digests) == 1, 'Attached ELIA checkpoint requires trusted-digest.txt'
    prior = prior_checkpoints[0]
    prior_digest = prior_digests[0].read_text(encoding='utf-8').strip().lower()
    assert prior.read_bytes()[:len(ENVELOPE_MAGIC)] == ENVELOPE_MAGIC, 'Refusing plaintext legacy state'
    result = subprocess.run([sys.executable, '-m', 'elia', '--checkpoint-restore', str(prior), '--expected-checkpoint-digest', prior_digest], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    assert result.returncode == 0, 'Encrypted checkpoint restore failed'
    print('Restored prior ELIA state:', prior_digest)
else:
    shutil.rmtree(STATE_DIR, ignore_errors=True)
    result = subprocess.run(['elia-bootstrap', '--cycles', '2'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    assert result.returncode == 0, 'Genesis bootstrap failed'
    print('Created fresh ELIA Genesis state.')

In [ ]:
# CPU-only organism proof. Every command must pass before model installation/loading.
checks = [
    ['elia-doctor'],
    ['elia-vitals'],
    [sys.executable, '-m', 'elia', '--verify'],
    [sys.executable, '-m', 'elia', '--status'],
    ['elia-supervisor', '--dry-run'],
]
for cmd in checks:
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('$', ' '.join(cmd))
    print(result.stdout[-12000:])
    assert result.returncode == 0, f'CPU gate failed: {cmd}'
print('CPU organism gates: GREEN')

In [ ]:
# Confirm a supported CUDA GPU with an actual CUDA operation, not just device discovery.
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'
name = torch.cuda.get_device_name(0)
print(name)
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1024**3:.2f} GiB')
x = torch.ones(1, device='cuda')
assert float(x.item()) == 1.0
del x
torch.cuda.empty_cache()
print('CUDA execution gate: GREEN')

In [ ]:
# Install the pinned Qwen 4-bit backend only after all CPU/CUDA gates pass.
!python -m pip install -q -e '.[gpu]'

In [ ]:
# One bounded real cognition cycle. --force-wake bypasses only the scheduler timestamp.
result = subprocess.run([sys.executable, '-m', 'elia', '--force-wake', '--cycles', '1'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
assert result.returncode == 0, 'Real ELIA cognition cycle failed'

In [ ]:
# Verify accepted post-cognition state and export a mandatory encrypted checkpoint.
for cmd in ([sys.executable, '-m', 'elia', '--verify'], ['elia-vitals']):
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    assert result.returncode == 0, f'Post-cognition gate failed: {cmd}'

export = subprocess.run([sys.executable, '-m', 'elia', '--checkpoint-export', str(CHECKPOINT)], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(export.stdout)
assert export.returncode == 0, 'Encrypted checkpoint export failed'
payload = parse_cli_json(export.stdout)
digest = payload['checkpoint']['digest']
DIGEST_FILE.write_text(digest + '\n', encoding='utf-8')
assert CHECKPOINT.read_bytes()[:len(ENVELOPE_MAGIC)] == ENVELOPE_MAGIC, 'Checkpoint is not encrypted'
print('Encrypted checkpoint:', CHECKPOINT, 'digest:', digest)

In [ ]:
# Independent restore proof: destroy an unrelated target state, restore the exported checkpoint, verify it.
shutil.rmtree(RESTORE_PROOF, ignore_errors=True)
proof_env = os.environ.copy()
proof_env['ELIA_STATE_DIR'] = str(RESTORE_PROOF)
restore = subprocess.run([sys.executable, '-m', 'elia', '--checkpoint-restore', str(CHECKPOINT), '--expected-checkpoint-digest', digest], env=proof_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(restore.stdout)
assert restore.returncode == 0, 'Independent restore proof failed'
verify = subprocess.run([sys.executable, '-m', 'elia', '--verify'], env=proof_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(verify.stdout)
assert verify.returncode == 0, 'Restored Chronicle verification failed'
print('RESTORE PROOF: GREEN')

## Successful completion

If every cell is green, this session has demonstrated installation, restore-or-genesis, CPU viability, real CUDA execution, one Qwen-backed ELIA cycle, post-state integrity, encrypted persistence, and independent restore.

`/kaggle/working/elia-genesis.eliacp` and `/kaggle/working/trusted-digest.txt` are the handoff artifacts. For unattended wake relay, use one persistent private Kaggle kernel ID with `ELIA_CHECKPOINT_KEY` and `ELIA_CHECKPOINT_ENCRYPTION_KEY` attached in Kaggle Secrets; Kaggle kernel metadata itself cannot attach user secrets.